# OLMo ICL Developmental Sweep

This notebook analyzes **when in-context learning (ICL) behavior emerges during OLMo-2 Stage-1 pretraining**.

The workflow is intentionally staged:

1. **Freeze and audit the datasets** before loading any model checkpoint.
2. **Build one fixed, balanced ICL assessment subset** that is reused for every checkpoint.
3. **Discover the actual OLMo Stage-1 revisions** available on Hugging Face.
4. **Run an endpoint sanity test** comparing the earliest and latest Stage-1 checkpoints.
5. **Compare endpoint behavior at the prompt level**, including exact paired significance tests.
6. **Prepare the developmental checkpoint sweep** that will locate the emergence window.
7. Use the resulting window to decide which checkpoints merit the more expensive RI analysis.

The notebook saves individual predictions as JSONL and aggregate tables as CSV so results can be reproduced without rerunning every model.

# 1. Setup

Install the packages used in this notebook, mount Google Drive, and define the project configuration.

In [ ]:
!pip -q install transformers huggingface_hub pandas numpy scipy statsmodels tqdm accelerate

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import gc
import hashlib
import json
import math
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import list_repo_refs
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# ------------------------------------------------------------------
# Project configuration
# ------------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/NLP_Project/olmo_sih_dynamics")
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results" / "icl"

MODEL_NAME = "allenai/OLMo-2-1124-7B"

ICL_PATH = DATA_DIR / "icl_stream.jsonl"
RI_RAW_PATH = DATA_DIR / "ri_agenda_stream.jsonl"
RI_SAFE_PATH = DATA_DIR / "ri_agenda_olmo_safe.jsonl"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TASKS = {
    "binary_fruit_month",
    "binary_furniture_profession",
    "four_class",
    "nine_class",
}

EXPECTED_SHOTS = {0, 1, 2, 3, 4, 5, 10, 20}

ASSESSMENT_EXAMPLES_PER_CONDITION = 50

print("Project root:", ROOT)
print("Results directory:", RESULTS_DIR)
print("Model:", MODEL_NAME)

## 1.1 Shared file utilities

In [ ]:
def load_jsonl(path):
    """Load a JSONL file into a list of dictionaries."""
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def write_jsonl(rows, path):
    """Write dictionaries to JSONL."""
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")


def sha256_file(path):
    """Compute a SHA-256 digest without loading the full file into memory."""
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

## 1.2 Load the OLMo tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

space_ids = tokenizer.encode(
    " ",
    add_special_tokens=False,
)

if len(space_ids) != 1:
    raise ValueError(
        f"Expected a single tokenizer ID for one space, got: {space_ids}"
    )

SPACE_TOKEN_ID = space_ids[0]

print("Tokenizer loaded.")
print("Vocabulary size:", len(tokenizer))
print("Whitespace token ID:", SPACE_TOKEN_ID)

# 2. Freeze and Audit the Datasets

All checkpoints must be evaluated on **identical data**. Before any model weights are loaded, this section records file hashes and verifies the structure of the ICL and RI datasets.

The ICL audit checks:

- expected tasks and shot counts;
- unique prompt IDs;
- valid gold labels;
- correct demonstration counts;
- prompt termination with the expected whitespace token;
- one-token answer labels under the OLMo tokenizer;
- no query pair duplicated among its demonstrations;
- complete demonstration-class coverage whenever the number of shots permits it.

The RI dataset is also checked to confirm that every retained row passed the token-safety filter.

## 2.1 Record dataset hashes

In [ ]:
DATA_FILES = [
    ICL_PATH,
    RI_RAW_PATH,
    RI_SAFE_PATH,
]

dataset_hashes = {}

for path in DATA_FILES:
    digest = sha256_file(path)
    dataset_hashes[path.name] = digest

    print(path.name)
    print("SHA256:", digest)
    print()

## 2.2 Load the frozen datasets

In [ ]:
icl_rows = load_jsonl(ICL_PATH)
ri_rows = load_jsonl(RI_SAFE_PATH)

print("ICL rows:", len(icl_rows))
print("RI rows:", len(ri_rows))

print("\nICL tasks:")
print(Counter(row["task"] for row in icl_rows))

print("\nShot counts:")
print(Counter(row["shots"] for row in icl_rows))

print("\nRI relation counts:")
print(Counter(row["relation"] for row in ri_rows))

print(
    "\nAll RI rows token-safe:",
    all(row["token_safe"] for row in ri_rows),
)

## 2.3 ICL row-level integrity audit

In [ ]:
def audit_icl_row(row, tokenizer):
    """Return all integrity errors detected for one generated ICL row."""
    errors = []

    # Gold label must be one of the task's allowed labels.
    if row["gold_label"] not in row["allowed_labels"]:
        errors.append("gold_not_allowed")

    # The serialized number of demonstrations must match the shot condition.
    if len(row["demos"]) != row["shots"]:
        errors.append("wrong_demo_count")

    # The prompt must end with the standalone whitespace token used by
    # the data generator before the answer label.
    prompt_ids = tokenizer.encode(
        row["prompt"],
        add_special_tokens=False,
    )

    if not prompt_ids or prompt_ids[-1] != SPACE_TOKEN_ID:
        errors.append("prompt_not_space_terminated")

    # Each legal answer must correspond to exactly one tokenizer token.
    for label in row["allowed_labels"]:
        label_ids = tokenizer.encode(
            label,
            add_special_tokens=False,
        )

        if len(label_ids) != 1:
            errors.append(f"multitoken_label_{label}")

    # The query lexical pair must not appear in its own demonstrations.
    query_pair = (
        row["query"]["left"],
        row["query"]["right"],
    )

    demo_pairs = {
        (demo["left"], demo["right"])
        for demo in row["demos"]
    }

    if query_pair in demo_pairs:
        errors.append("query_in_demos")

    # If enough demonstrations are available, every label class should
    # occur at least once.
    n_classes = len(row["allowed_labels"])

    if row["shots"] >= n_classes:
        demo_labels = {
            demo["label"]
            for demo in row["demos"]
        }

        if demo_labels != set(row["allowed_labels"]):
            errors.append("missing_demo_class")

    return {
        "id": row["id"],
        "errors": errors,
        "prompt_tokens": len(prompt_ids),
    }

In [ ]:
# Dataset-level structural checks.
if len({row["id"] for row in icl_rows}) != len(icl_rows):
    raise AssertionError("Duplicate ICL IDs found.")

if set(row["task"] for row in icl_rows) != EXPECTED_TASKS:
    raise AssertionError("Unexpected task set.")

if set(row["shots"] for row in icl_rows) != EXPECTED_SHOTS:
    raise AssertionError("Unexpected shot-count set.")

# Row-level checks.
audit_results = [
    audit_icl_row(row, tokenizer)
    for row in icl_rows
]

bad_rows = [
    result
    for result in audit_results
    if result["errors"]
]

print("Rows checked:", len(audit_results))
print("Rows with problems:", len(bad_rows))

if bad_rows:
    display(pd.DataFrame(bad_rows[:20]))
else:
    print("ALL ICL DATASET INTEGRITY CHECKS PASSED")

## 2.4 Diagnostic dataset statistics

In [ ]:
# Prompt-length distribution.
length_df = pd.DataFrame([
    {
        "task": row["task"],
        "shots": row["shots"],
        "tokens": audit["prompt_tokens"],
    }
    for row, audit in zip(icl_rows, audit_results)
])

length_summary = (
    length_df
    .groupby(["task", "shots"])["tokens"]
    .agg(["min", "mean", "median", "max"])
)

display(length_summary)

In [ ]:
# Gold-label distribution within every task × shot condition.
gold_balance = (
    pd.DataFrame([
        {
            "task": row["task"],
            "shots": row["shots"],
            "gold": row["gold_label"],
        }
        for row in icl_rows
    ])
    .groupby(["task", "shots", "gold"])
    .size()
    .rename("count")
    .reset_index()
)

display(gold_balance)

In [ ]:
def demo_class_coverage(row):
    """Number of distinct labels represented in a prompt's demonstrations."""
    return len({
        demo["label"]
        for demo in row["demos"]
    })


coverage_df = pd.DataFrame([
    {
        "task": row["task"],
        "shots": row["shots"],
        "n_demo_classes": demo_class_coverage(row),
        "n_possible_classes": len(row["allowed_labels"]),
    }
    for row in icl_rows
])

coverage_summary = (
    coverage_df
    .groupby(["task", "shots"])
    .agg(
        min_demo_classes=("n_demo_classes", "min"),
        mean_demo_classes=("n_demo_classes", "mean"),
        max_demo_classes=("n_demo_classes", "max"),
        possible_classes=("n_possible_classes", "first"),
    )
)

display(coverage_summary)

## 2.5 Final frozen-dataset fingerprint

In [ ]:
icl_hash = sha256_file(ICL_PATH)

print("Frozen ICL file:", ICL_PATH)
print("Rows:", len(icl_rows))
print("SHA256:", icl_hash)

# 3. Build the Fixed ICL Assessment Subset

The full generated dataset contains 100 examples for every task × shot condition. The broad developmental sweep uses **50 examples per condition**, balanced as evenly as possible across gold labels.

The subset is selected deterministically from sorted IDs and is constructed **once** before any checkpoint evaluation. This guarantees that all checkpoints receive the exact same prompts.

In [ ]:
def stratified_subset(rows, n_per_condition=50):
    """Select a deterministic label-balanced subset for each task × shot condition."""
    grouped = defaultdict(list)

    for row in rows:
        key = (
            row["task"],
            row["shots"],
            row["gold_label"],
        )
        grouped[key].append(row)

    conditions = defaultdict(dict)

    for (task, shots, label), group in grouped.items():
        conditions[(task, shots)][label] = sorted(
            group,
            key=lambda x: x["id"],
        )

    selected = []

    for (task, shots), label_groups in sorted(conditions.items()):
        labels = sorted(label_groups)

        base = n_per_condition // len(labels)
        remainder = n_per_condition % len(labels)

        for i, label in enumerate(labels):
            n = base + (1 if i < remainder else 0)
            chosen = label_groups[label][:n]

            if len(chosen) != n:
                raise ValueError(
                    f"Not enough examples for {task}, shots={shots}, "
                    f"label={label}: needed {n}, found {len(chosen)}"
                )

            selected.extend(chosen)

    return selected

In [ ]:
icl_assessment = stratified_subset(
    icl_rows,
    n_per_condition=ASSESSMENT_EXAMPLES_PER_CONDITION,
)

assessment_df = pd.DataFrame([
    {
        "id": row["id"],
        "task": row["task"],
        "shots": row["shots"],
        "gold": row["gold_label"],
    }
    for row in icl_assessment
])

print("Assessment examples:", len(icl_assessment))

display(
    assessment_df
    .groupby(["task", "shots"])
    .size()
    .rename("n")
    .reset_index()
)

display(
    assessment_df
    .groupby(["task", "shots", "gold"])
    .size()
    .rename("n")
    .reset_index()
)

# 4. Discover the Available OLMo Stage-1 Checkpoints

Intermediate checkpoints are Hugging Face revisions whose names follow the form:

`stage1-step<STEP>-tokens<TOKENS>B`

Instead of hard-coding the endpoint revisions, this section queries the repository and derives the earliest and latest available Stage-1 checkpoints.

In [ ]:
refs = list_repo_refs(MODEL_NAME)

stage1_revisions = [
    branch.name
    for branch in refs.branches
    if branch.name.startswith("stage1-step")
]

print("Stage-1 revisions found:", len(stage1_revisions))

In [ ]:
CHECKPOINT_RE = re.compile(
    r"stage1-step(\d+)-tokens(\d+)B"
)

checkpoint_info = []

for revision in stage1_revisions:
    match = CHECKPOINT_RE.fullmatch(revision)

    if match is None:
        continue

    checkpoint_info.append({
        "revision": revision,
        "step": int(match.group(1)),
        "tokens_B": int(match.group(2)),
    })

checkpoint_info = sorted(
    checkpoint_info,
    key=lambda x: (x["tokens_B"], x["step"]),
)

if not checkpoint_info:
    raise RuntimeError("No parsable Stage-1 checkpoints were found.")

checkpoint_df = pd.DataFrame(checkpoint_info)

EARLY_CHECKPOINT = checkpoint_info[0]["revision"]
FINAL_STAGE1_CHECKPOINT = checkpoint_info[-1]["revision"]

print("Parsed Stage-1 checkpoints:", len(checkpoint_info))
print("\nEarliest checkpoint:", EARLY_CHECKPOINT)
print("Latest checkpoint:", FINAL_STAGE1_CHECKPOINT)

display(checkpoint_df.head(15))
display(checkpoint_df.tail(15))

# 5. ICL Evaluation Utilities

For each prompt, the evaluator records both **constrained** and **unconstrained** behavior.

### Constrained metrics

Only legal task labels compete:

- predicted label;
- accuracy;
- gold-vs-best-wrong logit margin.

### Vocabulary-wide metrics

The model is also evaluated without restricting the vocabulary:

- whether the globally most likely token is a legal label;
- whether that token is the gold label;
- total probability mass assigned to legal labels;
- probability assigned to the gold label.

Together these metrics distinguish learning the task from merely learning the expected output format.

In [ ]:
# Verify once that labels 0-8 are individual OLMo tokens.
LABEL_TOKEN_IDS = {}

for label in map(str, range(9)):
    ids = tokenizer.encode(
        label,
        add_special_tokens=False,
    )

    if len(ids) != 1:
        raise ValueError(
            f"Label {label!r} is not a single token: {ids}"
        )

    LABEL_TOKEN_IDS[label] = ids[0]

print("Label token IDs:", LABEL_TOKEN_IDS)

In [ ]:
def evaluate_icl_example(model, tokenizer, row):
    """Evaluate one ICL prompt from the next-token logits."""
    inputs = tokenizer(
        row["prompt"],
        return_tensors="pt",
        add_special_tokens=False,
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model(**inputs)

    next_logits = outputs.logits[0, -1].float()

    allowed = row["allowed_labels"]
    gold = row["gold_label"]

    class_logits = {
        label: next_logits[LABEL_TOKEN_IDS[label]].item()
        for label in allowed
    }

    # --------------------------------------------------------------
    # 1. Constrained prediction among legal labels
    # --------------------------------------------------------------
    constrained_prediction = max(
        class_logits,
        key=class_logits.get,
    )

    constrained_correct = (
        constrained_prediction == gold
    )

    best_wrong_logit = max(
        score
        for label, score in class_logits.items()
        if label != gold
    )

    constrained_margin = (
        class_logits[gold] - best_wrong_logit
    )

    # --------------------------------------------------------------
    # 2. Vocabulary-wide next-token prediction
    # --------------------------------------------------------------
    global_top_id = int(
        torch.argmax(next_logits).item()
    )

    global_top_token = tokenizer.convert_ids_to_tokens(
        [global_top_id]
    )[0]

    allowed_ids = {
        LABEL_TOKEN_IDS[label]
        for label in allowed
    }

    gold_id = LABEL_TOKEN_IDS[gold]

    format_valid = global_top_id in allowed_ids
    unconstrained_correct = global_top_id == gold_id

    # --------------------------------------------------------------
    # 3. Probability mass on legal labels
    # --------------------------------------------------------------
    log_probs = torch.log_softmax(
        next_logits,
        dim=-1,
    )

    allowed_log_probs = torch.stack([
        log_probs[LABEL_TOKEN_IDS[label]]
        for label in allowed
    ])

    label_probability_mass = torch.exp(
        torch.logsumexp(
            allowed_log_probs,
            dim=0,
        )
    ).item()

    gold_probability = torch.exp(
        log_probs[gold_id]
    ).item()

    return {
        "gold": gold,
        "constrained_prediction": constrained_prediction,
        "constrained_correct": bool(constrained_correct),
        "constrained_margin": float(constrained_margin),
        "global_top_token_id": global_top_id,
        "global_top_token": global_top_token,
        "format_valid": bool(format_valid),
        "unconstrained_correct": bool(unconstrained_correct),
        "label_probability_mass": float(label_probability_mass),
        "gold_probability": float(gold_probability),
        "class_logits": class_logits,
    }

In [ ]:
def preferred_dtype():
    """Choose a practical inference dtype for the current Colab GPU."""
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16

    if torch.cuda.is_available():
        return torch.float16

    return torch.float32


def load_checkpoint(revision):
    """Load one OLMo checkpoint for inference."""
    dtype = preferred_dtype()

    print("Loading:", revision)
    print("dtype:", dtype)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=revision,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )

    model.eval()
    return model


def release_model(model):
    """Release a loaded checkpoint before loading the next one."""
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def run_checkpoint(
    model,
    tokenizer,
    revision,
    examples,
    output_path,
):
    """Evaluate one checkpoint and save every prompt-level prediction."""
    results = []

    for row in tqdm(
        examples,
        desc=revision,
    ):
        result = evaluate_icl_example(
            model,
            tokenizer,
            row,
        )

        results.append({
            "checkpoint": revision,
            "id": row["id"],
            "task": row["task"],
            "shots": row["shots"],
            "trial": row["trial"],
            "gold": row["gold_label"],
            **result,
        })

    write_jsonl(results, output_path)

    print(
        f"Saved {len(results):,} predictions -> {output_path}"
    )

    return results

In [ ]:
def summarize_icl_results(results):
    """Aggregate prompt-level ICL results by task and shot count."""
    df = pd.DataFrame(results)

    summary = (
        df
        .groupby(["task", "shots"])
        .agg(
            n=("id", "count"),
            constrained_accuracy=("constrained_correct", "mean"),
            mean_margin=("constrained_margin", "mean"),
            format_accuracy=("format_valid", "mean"),
            unconstrained_accuracy=("unconstrained_correct", "mean"),
            mean_label_mass=("label_probability_mass", "mean"),
            mean_gold_probability=("gold_probability", "mean"),
        )
        .reset_index()
    )

    return df, summary


def compute_shot_gains(summary):
    """Compute selected high-shot minus low-shot changes for each task."""
    rows = []

    for task, group in summary.groupby("task"):
        by_shot = group.set_index("shots")

        def value(shots, column):
            if shots not in by_shot.index:
                return np.nan
            return float(by_shot.loc[shots, column])

        rows.append({
            "task": task,
            "acc_20_minus_0":
                value(20, "constrained_accuracy")
                - value(0, "constrained_accuracy"),
            "acc_20_minus_1":
                value(20, "constrained_accuracy")
                - value(1, "constrained_accuracy"),
            "acc_10_minus_1":
                value(10, "constrained_accuracy")
                - value(1, "constrained_accuracy"),
            "acc_5_minus_1":
                value(5, "constrained_accuracy")
                - value(1, "constrained_accuracy"),
            "margin_20_minus_0":
                value(20, "mean_margin")
                - value(0, "mean_margin"),
            "margin_20_minus_1":
                value(20, "mean_margin")
                - value(1, "mean_margin"),
            "margin_10_minus_1":
                value(10, "mean_margin")
                - value(1, "mean_margin"),
            "format_20_minus_0":
                value(20, "format_accuracy")
                - value(0, "format_accuracy"),
            "format_20_minus_1":
                value(20, "format_accuracy")
                - value(1, "format_accuracy"),
        })

    return pd.DataFrame(rows)

# 6. Endpoint Sanity Test

Before spending compute on many intermediate checkpoints, compare the **earliest** and **latest** Stage-1 checkpoints on exactly the same assessment prompts.

The key question is whether the mature Stage-1 model exhibits substantially more ICL behavior than the earliest model. If the endpoints are nearly indistinguishable, the task or evaluation should be investigated before running the full trajectory.

## 6.1 Evaluate the final Stage-1 checkpoint

In [ ]:
final_model = load_checkpoint(
    FINAL_STAGE1_CHECKPOINT
)

final_path = (
    RESULTS_DIR
    / f"{FINAL_STAGE1_CHECKPOINT}.jsonl"
)

final_results = run_checkpoint(
    final_model,
    tokenizer,
    FINAL_STAGE1_CHECKPOINT,
    icl_assessment,
    final_path,
)

df_final, summary_final = summarize_icl_results(
    final_results
)

final_gains = compute_shot_gains(
    summary_final
)

display(summary_final)
display(final_gains)

In [ ]:
summary_final.to_csv(
    RESULTS_DIR / f"{FINAL_STAGE1_CHECKPOINT}_summary.csv",
    index=False,
)

final_gains.to_csv(
    RESULTS_DIR / f"{FINAL_STAGE1_CHECKPOINT}_shot_gains.csv",
    index=False,
)

In [ ]:
release_model(final_model)

## 6.2 Evaluate the earliest Stage-1 checkpoint

In [ ]:
early_model = load_checkpoint(
    EARLY_CHECKPOINT
)

# Quick one-example smoke test before the full run.
smoke_test = evaluate_icl_example(
    early_model,
    tokenizer,
    icl_assessment[0],
)

print(smoke_test)

In [ ]:
early_path = (
    RESULTS_DIR
    / f"{EARLY_CHECKPOINT}.jsonl"
)

early_results = run_checkpoint(
    early_model,
    tokenizer,
    EARLY_CHECKPOINT,
    icl_assessment,
    early_path,
)

df_early, summary_early = summarize_icl_results(
    early_results
)

early_gains = compute_shot_gains(
    summary_early
)

display(summary_early)
display(early_gains)

summary_early.to_csv(
    RESULTS_DIR / f"{EARLY_CHECKPOINT}_summary.csv",
    index=False,
)

early_gains.to_csv(
    RESULTS_DIR / f"{EARLY_CHECKPOINT}_shot_gains.csv",
    index=False,
)

## 6.3 Prompt-level endpoint comparison

In [ ]:
paired = df_early.merge(
    df_final,
    on="id",
    suffixes=("_early", "_final"),
    validate="one_to_one",
)

print("Paired examples:", len(paired))

paired["accuracy_delta"] = (
    paired["constrained_correct_final"].astype(int)
    - paired["constrained_correct_early"].astype(int)
)

paired["margin_delta"] = (
    paired["constrained_margin_final"]
    - paired["constrained_margin_early"]
)

paired["format_delta"] = (
    paired["format_valid_final"].astype(int)
    - paired["format_valid_early"].astype(int)
)

paired["label_mass_delta"] = (
    paired["label_probability_mass_final"]
    - paired["label_probability_mass_early"]
)

paired["gold_probability_delta"] = (
    paired["gold_probability_final"]
    - paired["gold_probability_early"]
)

In [ ]:
# Focus the endpoint diagnostic on the highest-shot condition.
paired_20 = paired[
    paired["shots_early"] == 20
].copy()

endpoint_20 = (
    paired_20
    .groupby("task_early")
    .agg(
        n=("id", "count"),
        accuracy_early=("constrained_correct_early", "mean"),
        accuracy_final=("constrained_correct_final", "mean"),
        mean_accuracy_delta=("accuracy_delta", "mean"),
        margin_early=("constrained_margin_early", "mean"),
        margin_final=("constrained_margin_final", "mean"),
        mean_margin_delta=("margin_delta", "mean"),
        label_mass_early=("label_probability_mass_early", "mean"),
        label_mass_final=("label_probability_mass_final", "mean"),
        mean_label_mass_delta=("label_mass_delta", "mean"),
        format_early=("format_valid_early", "mean"),
        format_final=("format_valid_final", "mean"),
    )
    .reset_index()
    .rename(columns={"task_early": "task"})
)

display(endpoint_20)

endpoint_20.to_csv(
    RESULTS_DIR / "endpoint_20shot_comparison.csv",
    index=False,
)

## 6.4 Correctness transitions and paired significance test

In [ ]:
def transition_counts(group):
    """Count prompt-level correctness transitions from early to final checkpoint."""
    early = group["constrained_correct_early"].astype(bool)
    final = group["constrained_correct_final"].astype(bool)

    return pd.Series({
        "wrong_to_wrong": int((~early & ~final).sum()),
        "wrong_to_correct": int((~early & final).sum()),
        "correct_to_wrong": int((early & ~final).sum()),
        "correct_to_correct": int((early & final).sum()),
    })


transitions_20 = (
    paired_20
    .groupby("task_early")
    .apply(transition_counts)
    .reset_index()
    .rename(columns={"task_early": "task"})
)

display(transitions_20)

In [ ]:
# Exact McNemar test:
# conditional on discordant pairs, test whether improvements and regressions
# are equally likely. Then control family-wise error across the four tasks
# using Holm's correction.

test_rows = []

for _, row in transitions_20.iterrows():
    improved = int(row["wrong_to_correct"])
    regressed = int(row["correct_to_wrong"])
    discordant = improved + regressed

    if discordant == 0:
        p_value = 1.0
    else:
        p_value = binomtest(
            min(improved, regressed),
            n=discordant,
            p=0.5,
            alternative="two-sided",
        ).pvalue

    test_rows.append({
        "task": row["task"],
        "wrong_to_correct": improved,
        "correct_to_wrong": regressed,
        "discordant_pairs": discordant,
        "raw_p": p_value,
    })

endpoint_tests = pd.DataFrame(test_rows)

reject, p_holm, _, _ = multipletests(
    endpoint_tests["raw_p"],
    alpha=0.05,
    method="holm",
)

endpoint_tests["holm_p"] = p_holm
endpoint_tests["holm_significant_0.05"] = reject

display(endpoint_tests)

endpoint_tests.to_csv(
    RESULTS_DIR / "endpoint_20shot_paired_tests.csv",
    index=False,
)

In [ ]:
release_model(early_model)

# 7. Developmental Checkpoint Sweep

This section runs the full frozen ICL assessment across the early-dense Stage-1 checkpoint schedule:

`1, 3, 5, 9, 13, 17, 26, 38, 80, 160, 320, 640, 1500, 3896` billion tokens.

with new tokens added at:
`700, 850, 900`
after the inital RI experiment.


It deliberately reuses the tokenizer, frozen assessment set, checkpoint inventory, and evaluation definitions created earlier in this notebook rather than duplicating them in a separate script.

The runner is resumable: complete checkpoints are loaded from their saved JSONL files, partial checkpoints continue from the missing prompt IDs, and every new batch is flushed to Google Drive immediately.

`RUN_NEW_CHECKPOINTS = False` is the safe organization/testing mode. It reads and summarizes existing results without downloading unfinished checkpoints. Change it to `True` only when you actually want to resume or reproduce the expensive sweep.


## 7.1 Sweep configuration and checkpoint plan


In [ ]:
# Section 7 depends on objects defined earlier in this notebook.
required = [
    "MODEL_NAME", "RESULTS_DIR", "tokenizer", "LABEL_TOKEN_IDS",
    "icl_assessment", "checkpoint_info", "summarize_icl_results",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the earlier notebook sections first. Missing: " + ", ".join(missing)
    )

TARGET_TOKENS_B = [1, 3, 5, 9, 13, 17, 26, 38, 80, 160, 320, 640, 1500, 3896]

# Extra checkpoints around the 3B -> 5B ICL transition.
#
# These are inserted explicitly because token-count based checkpoint
# selection cannot distinguish:
#   step600 and step700  -> both labelled ~3B
#   step850 and step900  -> both labelled ~4B
MANUAL_CHECKPOINTS = [
    {
        "revision": "stage1-step700-tokens3B",
        "step": 700,
        "tokens_B": 3,
    },
    {
        "revision": "stage1-step850-tokens4B",
        "step": 850,
        "tokens_B": 4,
    },
    {
        "revision": "stage1-step900-tokens4B",
        "step": 900,
        "tokens_B": 4,
    },
]

BATCH_SIZE = 4

# False = verify/rebuild summaries from existing files only.
# True  = download/resume any checkpoint whose 1,600 predictions are incomplete.
RUN_NEW_CHECKPOINTS = True

MODEL_CACHE_DIR = Path("/content/olmo_checkpoint_cache")
XET_CACHE_DIR = Path("/root/.cache/huggingface/xet")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

dtype = preferred_dtype()
assert torch.cuda.is_available(), "Use a GPU runtime before running new checkpoints."

def nearest_checkpoint(target_tokens_B):
    return min(
        checkpoint_info,
        key=lambda x: (abs(x["tokens_B"] - target_tokens_B), x["step"]),
    )

selected_checkpoints, seen = [], set()
for target in TARGET_TOKENS_B:
    cp = nearest_checkpoint(target)
    if cp["revision"] not in seen:
        selected_checkpoints.append(cp)
        seen.add(cp["revision"])

for cp in MANUAL_CHECKPOINTS:
    if cp["revision"] not in seen:
        selected_checkpoints.append(cp)
        seen.add(cp["revision"])


checkpoint_manifest = pd.DataFrame(selected_checkpoints)
checkpoint_manifest.to_csv(RESULTS_DIR / "selected_checkpoint_manifest.csv", index=False)

print("Results directory:", RESULTS_DIR)
print("GPU:", torch.cuda.get_device_name(0))
print("dtype:", dtype)
print("Examples per checkpoint:", len(icl_assessment))
print("Run unfinished checkpoints:", RUN_NEW_CHECKPOINTS)
display(checkpoint_manifest)


## 7.2 Batched evaluator

This is the batched equivalent of the single-example evaluator used for the endpoint test. Right padding is ignored when the next-token position is selected from the attention mask. All eight shot counts and all four tasks are retained.


In [ ]:
def evaluate_icl_batch(model, tokenizer, batch_rows):
    prompts = [row["prompt"] for row in batch_rows]
    encoded = tokenizer(
        prompts, return_tensors="pt", padding=True, add_special_tokens=False
    )
    encoded = {key: value.to(model.device) for key, value in encoded.items()}

    with torch.inference_mode():
        outputs = model(**encoded)

    last_positions = encoded["attention_mask"].sum(dim=1) - 1
    batch_indices = torch.arange(len(batch_rows), device=last_positions.device)
    next_logits_batch = outputs.logits[batch_indices, last_positions, :].float()

    results = []
    for i, row in enumerate(batch_rows):
        next_logits = next_logits_batch[i]
        allowed = row["allowed_labels"]
        gold = row["gold_label"]

        class_logits = {
            label: float(next_logits[LABEL_TOKEN_IDS[label]].item())
            for label in allowed
        }
        constrained_prediction = max(class_logits, key=class_logits.get)
        best_wrong_logit = max(
            score for label, score in class_logits.items() if label != gold
        )

        global_top_id = int(torch.argmax(next_logits).item())
        global_top_token = tokenizer.convert_ids_to_tokens([global_top_id])[0]
        allowed_ids = {LABEL_TOKEN_IDS[label] for label in allowed}
        gold_id = LABEL_TOKEN_IDS[gold]

        log_probs = torch.log_softmax(next_logits, dim=-1)
        allowed_log_probs = torch.stack(
            [log_probs[LABEL_TOKEN_IDS[label]] for label in allowed]
        )

        results.append({
            "gold": gold,
            "constrained_prediction": constrained_prediction,
            "constrained_correct": bool(constrained_prediction == gold),
            "constrained_margin": float(class_logits[gold] - best_wrong_logit),
            "global_top_token_id": global_top_id,
            "global_top_token": global_top_token,
            "format_valid": bool(global_top_id in allowed_ids),
            "unconstrained_correct": bool(global_top_id == gold_id),
            "label_probability_mass": float(
                torch.exp(torch.logsumexp(allowed_log_probs, dim=0)).item()
            ),
            "gold_probability": float(torch.exp(log_probs[gold_id]).item()),
            "class_logits": class_logits,
        })

    del outputs, next_logits_batch, encoded
    return results


## 7.3 Resumable result and checkpoint-cache utilities


In [ ]:
def load_jsonl_safe(path):
    if not path.exists():
        return []

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"WARNING: skipping malformed line {line_number} in {path.name}")
    return rows


def summarize_sweep_results(results, checkpoint):
    _, summary = summarize_icl_results(results)
    summary.insert(0, "checkpoint", checkpoint["revision"])
    summary.insert(1, "step", checkpoint["step"])
    summary.insert(2, "tokens_B", checkpoint["tokens_B"])
    return summary


def print_local_disk_usage(prefix="Disk"):
    usage = shutil.disk_usage("/")
    gb = 1024 ** 3
    print(
        f"{prefix}: {usage.used / gb:.1f} GB used, "
        f"{usage.free / gb:.1f} GB free of {usage.total / gb:.1f} GB"
    )


def clear_checkpoint_disk_cache():
    if MODEL_CACHE_DIR.exists():
        shutil.rmtree(MODEL_CACHE_DIR, ignore_errors=True)
    MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

    if XET_CACHE_DIR.exists():
        shutil.rmtree(XET_CACHE_DIR, ignore_errors=True)


def release_sweep_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    clear_checkpoint_disk_cache()


def load_sweep_checkpoint(revision, min_free_gb=50):
    clear_checkpoint_disk_cache()
    free_gb = shutil.disk_usage("/").free / (1024 ** 3)
    if free_gb < min_free_gb:
        raise RuntimeError(
            f"Only {free_gb:.1f} GB local disk is free; "
            f"{min_free_gb} GB is required before the next download."
        )

    print("\n" + "=" * 80)
    print("LOADING:", revision)
    print_local_disk_usage("Before checkpoint download")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=revision,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        cache_dir=str(MODEL_CACHE_DIR),
    )
    model.eval()
    return model


def run_checkpoint_resumable(model, checkpoint, examples, batch_size=BATCH_SIZE):
    revision = checkpoint["revision"]
    raw_path = RESULTS_DIR / f"{revision}.jsonl"
    summary_path = RESULTS_DIR / f"{revision}_summary.csv"

    existing = load_jsonl_safe(raw_path)
    processed_ids = {row["id"] for row in existing}
    expected_ids = {row["id"] for row in examples}
    remaining = [row for row in examples if row["id"] not in processed_ids]

    print(f"\nCheckpoint: {revision}")
    print(f"Already complete: {len(processed_ids)} / {len(examples)}")
    print("Remaining:", len(remaining))

    if remaining:
        with open(raw_path, "a", encoding="utf-8") as f:
            starts = range(0, len(remaining), batch_size)
            for start in tqdm(
                starts,
                total=math.ceil(len(remaining) / batch_size),
                desc=f'{checkpoint["tokens_B"]}B | {revision}',
            ):
                batch_rows = remaining[start:start + batch_size]
                batch_results = evaluate_icl_batch(model, tokenizer, batch_rows)

                for source_row, result in zip(batch_rows, batch_results):
                    output_row = {
                        "checkpoint": revision,
                        "step": checkpoint["step"],
                        "tokens_B": checkpoint["tokens_B"],
                        "id": source_row["id"],
                        "task": source_row["task"],
                        "shots": source_row["shots"],
                        "trial": source_row["trial"],
                        "gold": source_row["gold_label"],
                        **result,
                    }
                    f.write(json.dumps(output_row) + "\n")
                f.flush()

    results = load_jsonl_safe(raw_path)
    result_by_id = {
        row["id"]: row for row in results if row.get("id") in expected_ids
    }
    missing = expected_ids - set(result_by_id)
    if missing:
        raise RuntimeError(f"{revision}: {len(missing)} expected examples are missing.")

    results = list(result_by_id.values())
    summary = summarize_sweep_results(results, checkpoint)
    summary.to_csv(summary_path, index=False)

    print(f"Finished {revision}")
    display(
        summary.loc[
            summary["shots"] == 20,
            [
                "task", "constrained_accuracy", "mean_margin",
                "format_accuracy", "mean_label_mass", "mean_gold_probability",
            ],
        ]
    )
    return results, summary


## 7.4 Run or recover the developmental sweep

With `RUN_NEW_CHECKPOINTS = False`, this cell does **not** download model weights. It validates and summarizes any complete checkpoint JSONLs already stored in `RESULTS_DIR`.

With `RUN_NEW_CHECKPOINTS = True`, incomplete checkpoints are resumed automatically. The progress bars show both checkpoint-level and batch-level progress.


In [ ]:
all_summaries = []
expected_ids = {row["id"] for row in icl_assessment}
checkpoint_progress = tqdm(selected_checkpoints, desc="CHECKPOINTS")

for checkpoint in checkpoint_progress:
    revision = checkpoint["revision"]
    checkpoint_progress.set_postfix(checkpoint=f'{checkpoint["tokens_B"]}B')
    raw_path = RESULTS_DIR / f"{revision}.jsonl"

    existing = load_jsonl_safe(raw_path)
    result_by_id = {
        row["id"]: row for row in existing if row.get("id") in expected_ids
    }
    complete = expected_ids.issubset(result_by_id)

    if complete:
        print(f"\nALREADY COMPLETE: {revision} — no model load.")
        summary = summarize_sweep_results(list(result_by_id.values()), checkpoint)
        summary.to_csv(RESULTS_DIR / f"{revision}_summary.csv", index=False)
        all_summaries.append(summary)
        continue

    if not RUN_NEW_CHECKPOINTS:
        print(
            f"\nNOT RUN: {revision} — "
            f"{len(result_by_id)}/{len(expected_ids)} saved predictions."
        )
        continue

    model = None
    try:
        model = load_sweep_checkpoint(revision)

        # One-example preflight before committing to the checkpoint.
        preflight = evaluate_icl_batch(model, tokenizer, icl_assessment[:1])
        print("Preflight:", preflight[0])

        _, summary = run_checkpoint_resumable(
            model, checkpoint, icl_assessment, batch_size=BATCH_SIZE
        )
        all_summaries.append(summary)

    finally:
        if model is not None:
            release_sweep_model(model)
            model = None
        else:
            clear_checkpoint_disk_cache()


## 7.5 Combine and save the available trajectory


In [ ]:
if not all_summaries:
    raise RuntimeError(
        "No complete checkpoint results were found in RESULTS_DIR. "
        "Copy the saved JSONL files into the new project results directory "
        "or set RUN_NEW_CHECKPOINTS = True."
    )

trajectory = (
    pd.concat(all_summaries, ignore_index=True)
    .sort_values(["tokens_B", "task", "shots"])
    .reset_index(drop=True)
)
trajectory_20 = trajectory.loc[trajectory["shots"] == 20].copy()

trajectory_path = RESULTS_DIR / "icl_checkpoint_trajectory_all_shots.csv"
trajectory_20_path = RESULTS_DIR / "icl_checkpoint_trajectory_20shot.csv"

trajectory.to_csv(trajectory_path, index=False)
trajectory_20.to_csv(trajectory_20_path, index=False)

print("Complete checkpoints in trajectory:", trajectory["checkpoint"].nunique())
print("Saved:", trajectory_path)
print("Saved:", trajectory_20_path)

display(
    trajectory_20[
        [
            "tokens_B", "checkpoint", "task", "constrained_accuracy",
            "mean_margin", "format_accuracy", "mean_label_mass",
        ]
    ]
)


# Command line output from previous run (cleaned)

============================================================================================

SETUP
-----
Model: allenai/OLMo-2-1124-7B
GPU: NVIDIA L4
dtype: torch.bfloat16
Stage-1 revisions found / parsed: 928 / 928
Earliest checkpoint: stage1-step150-tokens1B
Latest checkpoint:   stage1-step928646-tokens3896B
Examples per checkpoint: 1600
Task × shot conditions: 32
Batch size used by the sweep: 4 (400 batches/checkpoint)

SELECTED CHECKPOINTS
--------------------
   1B | step    150 | stage1-step150-tokens1B
   3B | step    600 | stage1-step600-tokens3B
   5B | step   1000 | stage1-step1000-tokens5B
   9B | step   2000 | stage1-step2000-tokens9B
  13B | step   3000 | stage1-step3000-tokens13B
  17B | step   4000 | stage1-step4000-tokens17B
  26B | step   6000 | stage1-step6000-tokens26B
  38B | step   9000 | stage1-step9000-tokens38B
  80B | step  19000 | stage1-step19000-tokens80B
 160B | step  38000 | stage1-step38000-tokens160B
 319B | step  76000 | stage1-step76000-tokens319B
 638B | step 152000 | stage1-step152000-tokens638B
1498B | step 357000 | stage1-step357000-tokens1498B
3896B | step 928646 | stage1-step928646-tokens3896B

RUN HISTORY
-----------
Run 1:
  • Completed 1B, 3B, 5B, 9B, 13B, and 17B.
  • The 26B checkpoint download then failed because the default Hugging Face cache
    had only ~2.2 GB free while several ~4.6–5.0 GB shards still had to be written.
  • Error: File reconstruction error / Background writer channel closed.

Run 2 (resumed with disposable checkpoint-cache cleanup):
  • Loaded the six completed checkpoints from saved JSONL results.
  • Successfully completed 26B, 38B, 80B, 160B, 319B, 638B, 1498B, and 3896B.
  • Local disk returned to ~43.3 GB used / ~192.4 GB free after each cleanup.
  • Active resumed sweep finished 8/8 checkpoints in 2:05:42.

FINAL STATUS
------------
All 14 selected checkpoints completed.
Full trajectory saved as: icl_checkpoint_trajectory_all_shots.csv
20-shot trajectory saved as: icl_checkpoint_trajectory_20shot.csv

20-SHOT RESULTS
---------------
 Tokens  Task                              Acc    Margin  Format  LabelMass
----------------------------------------------------------------------------------
     1B  binary_fruit_month               0.46    -0.050    0.98      0.090
         binary_furniture_profession      0.44    -0.027    0.42      0.088
         four_class                       0.24    -0.268    1.00      0.140
         nine_class                       0.12    -0.636    0.98      0.185

     3B  binary_fruit_month               0.54     0.194    1.00      0.662
         binary_furniture_profession      0.48    -0.034    1.00      0.762
         four_class                       0.26    -0.721    1.00      0.611
         nine_class                       0.16    -0.954    1.00      0.675

     5B  binary_fruit_month               0.90     1.156    1.00      0.853
         binary_furniture_profession      0.84     0.589    1.00      0.896
         four_class                       0.52    -0.049    1.00      0.810
         nine_class                       0.28    -0.833    1.00      0.854

     9B  binary_fruit_month               0.78     0.788    1.00      0.906
         binary_furniture_profession      0.74     0.651    1.00      0.938
         four_class                       0.58     0.328    0.98      0.862
         nine_class                       0.30    -0.542    0.94      0.869

    13B  binary_fruit_month               0.90     0.966    1.00      0.936
         binary_furniture_profession      0.80     0.653    1.00      0.938
         four_class                       0.86     0.589    1.00      0.920
         nine_class                       0.42    -0.237    1.00      0.922

    17B  binary_fruit_month               0.94     0.739    1.00      0.961
         binary_furniture_profession      0.70     0.427    1.00      0.958
         four_class                       0.68     0.492    1.00      0.948
         nine_class                       0.38    -0.248    1.00      0.885

    26B  binary_fruit_month               0.86     0.706    1.00      0.955
         binary_furniture_profession      0.78     0.323    1.00      0.945
         four_class                       0.76     0.409    1.00      0.941
         nine_class                       0.42    -0.180    1.00      0.949

    38B  binary_fruit_month               0.94     1.417    1.00      0.983
         binary_furniture_profession      0.60     0.284    1.00      0.985
         four_class                       0.86     0.915    1.00      0.965
         nine_class                       0.60     0.128    1.00      0.951

    80B  binary_fruit_month               0.90     0.833    1.00      0.968
         binary_furniture_profession      0.60     0.121    1.00      0.973
         four_class                       0.78     0.422    1.00      0.954
         nine_class                       0.38    -0.186    1.00      0.960

   160B  binary_fruit_month               0.70     0.709    1.00      0.986
         binary_furniture_profession      0.62     0.395    1.00      0.988
         four_class                       0.64     0.355    1.00      0.969
         nine_class                       0.36    -0.230    1.00      0.950

   319B  binary_fruit_month               0.84     0.849    1.00      0.979
         binary_furniture_profession      0.58     0.179    1.00      0.982
         four_class                       0.62     0.179    1.00      0.967
         nine_class                       0.32    -0.266    1.00      0.959

   638B  binary_fruit_month               0.78     0.590    1.00      0.969
         binary_furniture_profession      0.52     0.009    1.00      0.989
         four_class                       0.72     0.421    1.00      0.966
         nine_class                       0.44    -0.119    1.00      0.957

  1498B  binary_fruit_month               0.70     0.334    1.00      0.980
         binary_furniture_profession      0.50     0.031    1.00      0.993
         four_class                       0.50     0.179    1.00      0.976
         nine_class                       0.32    -0.190    1.00      0.950

  3896B  binary_fruit_month               0.70     0.407    1.00      0.980
         binary_furniture_profession      0.50     0.033    1.00      0.991
         four_class                       0.58     0.198    1.00      0.974
         nine_class                       0.32    -0.219    1.00      0.954

NOTES
-----
• The huge HTML/CSS blocks in the original pasted output came from Colab rendering
  pandas DataFrames with display(...); they are presentation markup, not experiment data.
• The raw JSONL and CSV result files remain the authoritative data sources for analysis.
• This cleaned log intentionally keeps run status and the compact 20-shot summary only.


# 8. RI Sweep — Next Stage

The RI dataset is loaded and frozen at the beginning of this notebook, but the original notebook does not yet contain the model-side RI intervention/attention analysis.

The intended workflow is to **first locate the ICL emergence window with the cheaper behavioral sweep**, then run the more expensive RI analysis only on checkpoints immediately before, within, and after that window. Keeping the RI stage separate prevents unnecessary checkpoint loading and makes the relationship between behavioral emergence and the proposed semantic induction-head signal easier to interpret.